# LandingAI : section parsing

In [1]:
import json
import os
from pathlib import Path
from landingai_ade import LandingAIADE

client = LandingAIADE(apikey=os.environ.get("LANDINGAI_API_KEY"))

# Step 1: Parse
parse_response = client.parse(
    document=Path("/home/rajnikant/Github/filings/data/raw_filings/tsla-20250630-gen.pdf"),
    model="dpt-2-latest",
)

# Step 2: Section
section_response = client.section(
    markdown=parse_response.markdown,
    model="section-latest",
)

# Step 3: Build chunk lookup
chunk_map = {chunk.id: chunk for chunk in parse_response.chunks}
toc = section_response.table_of_contents

# Step 4: Get content for each section
def get_section_content(i, toc, parse_response):
    ref = toc[i].start_reference
    next_ref = toc[i + 1].start_reference if i + 1 < len(toc) else None
    section_chunks = []
    collecting = False
    for chunk in parse_response.chunks:
        if chunk.id == ref:
            collecting = True
        if collecting:
            if next_ref and chunk.id == next_ref:
                break
            section_chunks.append(chunk)
    return "\n\n".join(c.markdown for c in section_chunks).strip()

# Step 5: Build hierarchy
def build_hierarchy(toc, parse_response):
    root = []
    stack = []  # tracks (node, level)

    for i, section in enumerate(toc):
        node = {
            "section_number": section.section_number,
            "title": section.title,
            "level": section.level,
            "start_reference": section.start_reference,
            "content": get_section_content(i, toc, parse_response),
            "children": []
        }

        # Pop stack until we find a parent
        while stack and stack[-1]["level"] >= section.level:
            stack.pop()

        if stack:
            stack[-1]["children"].append(node)
        else:
            root.append(node)

        stack.append(node)

    return root

# Step 6: Build and save
hierarchy = build_hierarchy(toc, parse_response)

output_path = Path("output_hierarchy.json")
output_path.write_text(json.dumps(hierarchy, indent=2), encoding="utf-8")

print(f"Saved to {output_path}")

print(section_response)
print(parse_response)

APIStatusError: Error code: 402 - {'error': 'Credit quota exceeded (including pending jobs) for organization 5cb5w32ppe5c, please upgrade to a higher plan.'}

In [ ]:
chunk_map = {chunk.id: chunk for chunk in parse_response.chunks}
print(chunk_map)
print(parse_response.markdown)
print(section_response.table_of_contents_md)

{'63311c50-8834-482a-aa16-a7aae3cc4b40': Chunk(id='63311c50-8834-482a-aa16-a7aae3cc4b40', grounding=ChunkGrounding(box=ParseGroundingBox(bottom=0.1438426673412323, left=0.21562719345092773, right=0.7837883234024048, top=0.06943809986114502), page=0), markdown="<a id='63311c50-8834-482a-aa16-a7aae3cc4b40'></a>\n\nUNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-Q", type='text'), 'e31a26ce-8692-4769-8514-b23e5a91cb3f': Chunk(id='e31a26ce-8692-4769-8514-b23e5a91cb3f', grounding=ChunkGrounding(box=ParseGroundingBox(bottom=0.26984894275665283, left=0.015917539596557617, right=0.7920928001403809, top=0.14903271198272705), page=0), markdown="<a id='e31a26ce-8692-4769-8514-b23e5a91cb3f'></a>\n\n(Mark One)\noption QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934: [x]\nFor the quarterly period ended June 30, 2025\nOR\noption TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934: [ ]\nFor t